# Unidade 4 - Bloco prático da Aula 02: as 20 sementes

Treina os mesmos dois modelos 20 vezes, mudando apenas a semente do sorteio da divisão dos dados. Cada execução é um relatório final possível: compare a diferença entre as médias com o desvio observado dentro de um único modelo.

In [2]:
import time
import numpy as np

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import f1_score


# ============================================================
# 1. CONFIGURAÇÃO DO EXPERIMENTO
# ============================================================

N_SEMENTES = 20

print("=" * 70)
print("ESTABILIDADE DOS MODELOS EM DIFERENTES DIVISÕES DOS DADOS")
print("=" * 70)


# ============================================================
# 2. CRIANDO O DATASET
# ============================================================

X, y = make_classification(
    n_samples=1500,
    n_features=15,
    n_informative=6,

    # Dataset desbalanceado
    weights=[0.8, 0.2],

    # Pequena quantidade de ruído
    flip_y=0.03,

    random_state=0
)


print(f"\nAmostras = {X.shape[0]}")
print(f"Características = {X.shape[1]}")

print(
    f"Classe 0 = {(y == 0).sum()} "
    f"({(y == 0).mean() * 100:.1f}%)"
)

print(
    f"Classe 1 = {(y == 1).sum()} "
    f"({(y == 1).mean() * 100:.1f}%)"
)

print(
    f"\nSerão feitas {N_SEMENTES} divisões diferentes "
    f"entre treino e teste."
)


# ============================================================
# 3. LISTAS PARA GUARDAR OS RESULTADOS
# ============================================================

f1_rf = []
f1_gb = []

tempos_rf = []
tempos_gb = []


# ============================================================
# 4. REPETINDO O EXPERIMENTO COM DIFERENTES SEMENTES
# ============================================================

print("\n")
print("=" * 70)
print("EXECUTANDO AS DIVISÕES")
print("=" * 70)


for semente in range(N_SEMENTES):

    # --------------------------------------------------------
    # Cada semente gera uma divisão diferente
    # --------------------------------------------------------

    X_tr, X_te, y_tr, y_te = train_test_split(
        X,
        y,

        test_size=0.30,

        # Mantém a proporção das classes
        stratify=y,

        random_state=semente
    )


    # ========================================================
    # RANDOM FOREST
    # ========================================================

    rf = RandomForestClassifier(
        n_estimators=120,
        max_depth=8,

        random_state=42,
        n_jobs=-1
    )


    inicio = time.time()

    rf.fit(
        X_tr,
        y_tr
    )

    pred_rf = rf.predict(
        X_te
    )

    tempo_rf = (
        time.time() - inicio
    )

    resultado_rf = f1_score(
        y_te,
        pred_rf
    )


    # ========================================================
    # GRADIENT BOOSTING
    # ========================================================

    gb = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,

        random_state=42
    )


    inicio = time.time()

    gb.fit(
        X_tr,
        y_tr
    )

    pred_gb = gb.predict(
        X_te
    )

    tempo_gb = (
        time.time() - inicio
    )

    resultado_gb = f1_score(
        y_te,
        pred_gb
    )


    # ========================================================
    # GUARDANDO OS RESULTADOS
    # ========================================================

    f1_rf.append(
        resultado_rf
    )

    f1_gb.append(
        resultado_gb
    )

    tempos_rf.append(
        tempo_rf
    )

    tempos_gb.append(
        tempo_gb
    )


    # ========================================================
    # MOSTRANDO O RESULTADO DESTA SEMENTE
    # ========================================================

    diferenca = (
        resultado_gb
        - resultado_rf
    )


    if resultado_gb > resultado_rf:

        vencedor = "Boosting"

    elif resultado_rf > resultado_gb:

        vencedor = "Floresta"

    else:

        vencedor = "Empate"


    print(
        f"Semente {semente:02d}"
        f" | RF = {resultado_rf:.4f}"
        f" | GB = {resultado_gb:.4f}"
        f" | diferença = {diferenca:+.4f}"
        f" | vencedor = {vencedor}"
    )


# ============================================================
# 5. CONVERTENDO PARA NUMPY
# ============================================================

f1_rf = np.array(
    f1_rf
)

f1_gb = np.array(
    f1_gb
)

tempos_rf = np.array(
    tempos_rf
)

tempos_gb = np.array(
    tempos_gb
)


# ============================================================
# 6. RESULTADOS DA RANDOM FOREST
# ============================================================

print("\n")
print("=" * 70)
print("RANDOM FOREST")
print("=" * 70)


print(
    f"F1 médio = "
    f"{f1_rf.mean():.4f}"
)

print(
    f"Desvio padrão = "
    f"{f1_rf.std():.4f}"
)

print(
    f"Menor F1 = "
    f"{f1_rf.min():.4f}"
)

print(
    f"Maior F1 = "
    f"{f1_rf.max():.4f}"
)

print(
    f"Amplitude = "
    f"{f1_rf.max() - f1_rf.min():.4f}"
)

print(
    f"Tempo médio = "
    f"{tempos_rf.mean():.3f}s"
)


# ============================================================
# 7. RESULTADOS DO GRADIENT BOOSTING
# ============================================================

print("\n")
print("=" * 70)
print("GRADIENT BOOSTING")
print("=" * 70)


print(
    f"F1 médio = "
    f"{f1_gb.mean():.4f}"
)

print(
    f"Desvio padrão = "
    f"{f1_gb.std():.4f}"
)

print(
    f"Menor F1 = "
    f"{f1_gb.min():.4f}"
)

print(
    f"Maior F1 = "
    f"{f1_gb.max():.4f}"
)

print(
    f"Amplitude = "
    f"{f1_gb.max() - f1_gb.min():.4f}"
)

print(
    f"Tempo médio = "
    f"{tempos_gb.mean():.3f}s"
)


# ============================================================
# 8. COMPARAÇÃO DIRETA
# ============================================================

print("\n")
print("=" * 70)
print("COMPARAÇÃO DIRETA")
print("=" * 70)


vitorias_rf = (
    f1_rf > f1_gb
).sum()

vitorias_gb = (
    f1_gb > f1_rf
).sum()

empates = (
    f1_rf == f1_gb
).sum()


print(
    f"Random Forest venceu "
    f"{vitorias_rf} de {N_SEMENTES}"
)

print(
    f"Gradient Boosting venceu "
    f"{vitorias_gb} de {N_SEMENTES}"
)

print(
    f"Empates = "
    f"{empates}"
)


# ============================================================
# 9. DIFERENÇA ENTRE OS MODELOS
# ============================================================

diferencas = (
    f1_gb
    - f1_rf
)


print("\nDiferença GB - RF")

print(
    f"Média = "
    f"{diferencas.mean():+.4f}"
)

print(
    f"Maior vantagem do Boosting = "
    f"{diferencas.max():+.4f}"
)

print(
    f"Maior vantagem da Floresta = "
    f"{diferencas.min():+.4f}"
)


# ============================================================
# 10. QUAL MODELO FOI MAIS ESTÁVEL
# ============================================================

print("\n")
print("=" * 70)
print("ESTABILIDADE")
print("=" * 70)


if f1_rf.std() < f1_gb.std():

    print(
        "Random Forest foi mais estável."
    )

    print(
        f"Desvio RF = {f1_rf.std():.4f}"
    )

    print(
        f"Desvio GB = {f1_gb.std():.4f}"
    )

elif f1_gb.std() < f1_rf.std():

    print(
        "Gradient Boosting foi mais estável."
    )

    print(
        f"Desvio GB = {f1_gb.std():.4f}"
    )

    print(
        f"Desvio RF = {f1_rf.std():.4f}"
    )

else:

    print(
        "Os dois modelos apresentaram "
        "o mesmo desvio padrão."
    )


# ============================================================
# 11. CONCLUSÃO AUTOMÁTICA
# ============================================================

print("\n")
print("=" * 70)
print("CONCLUSÃO")
print("=" * 70)


if f1_gb.mean() > f1_rf.mean():

    melhor_media = "Gradient Boosting"

else:

    melhor_media = "Random Forest"


if f1_gb.std() < f1_rf.std():

    mais_estavel = "Gradient Boosting"

else:

    mais_estavel = "Random Forest"


print(
    f"Maior F1 médio = "
    f"{melhor_media}"
)

print(
    f"Modelo mais estável = "
    f"{mais_estavel}"
)

ESTABILIDADE DOS MODELOS EM DIFERENTES DIVISÕES DOS DADOS

Amostras = 1500
Características = 15
Classe 0 = 1182 (78.8%)
Classe 1 = 318 (21.2%)

Serão feitas 20 divisões diferentes entre treino e teste.


EXECUTANDO AS DIVISÕES
Semente 00 | RF = 0.7205 | GB = 0.7886 | diferença = +0.0681 | vencedor = Boosting
Semente 01 | RF = 0.7826 | GB = 0.7882 | diferença = +0.0056 | vencedor = Boosting
Semente 02 | RF = 0.7665 | GB = 0.7528 | diferença = -0.0137 | vencedor = Floresta
Semente 03 | RF = 0.7702 | GB = 0.7456 | diferença = -0.0246 | vencedor = Floresta
Semente 04 | RF = 0.7950 | GB = 0.8276 | diferença = +0.0326 | vencedor = Boosting
Semente 05 | RF = 0.7470 | GB = 0.7821 | diferença = +0.0351 | vencedor = Boosting
Semente 06 | RF = 0.7468 | GB = 0.7976 | diferença = +0.0508 | vencedor = Boosting
Semente 07 | RF = 0.7006 | GB = 0.7529 | diferença = +0.0523 | vencedor = Boosting
Semente 08 | RF = 0.7643 | GB = 0.8046 | diferença = +0.0403 | vencedor = Boosting
Semente 09 | RF = 0.7205 |